# Lesson 7 — Why Databases Exist + SQLite

**The bridge from Lessons 1, 2, 4, and 6.** Three ideas have been waiting for this lesson:

1. **Lesson 1:** a structure is a bundle of *properties* — a type per column, uniqueness, a
   legal set of values, a key that points at a record. We wrote that checklist down and
   promised to *honour* it.
2. **Lesson 2:** a file honours nothing. CSV forgot our types; a blank meant three different
   things; cleaning could only *repair* damage that was already on disk.
3. **Lesson 4:** a dictionary answers "have I seen this key?" in O(1). Hold that thought — a
   database turns that same trick into a *durable* guarantee.

Lesson 6 fetched JSON into tidy DataFrames; this lesson supplies the missing piece: **a schema is Lesson 1's property checklist, finally
enforced by software instead of by discipline.** You stop *hoping* every customer_id is unique
and every country is filled in; you *declare* it, and the database refuses — on write — to let
it break.

```
Business rules                   "every transaction belongs to a real customer"
        |
Schema             (Unit 2)      CREATE TABLE ... NOT NULL, PRIMARY KEY, FOREIGN KEY
        |
The database enforces it          it REJECTS the bad row instead of storing it
        |
Python talks to it (Unit 3)      sqlite3, to_sql / read_sql, the ? placeholder
        |
Right tool for the job (Unit 4)  database vs DataFrame
```

**Prerequisites and setup**

- Lesson 1: types, uniqueness, sets, keys — the property checklist.
- Lesson 2: `read_csv`, dtypes, missing values, `merge` and its orphan-row surprise.
- Lesson 4: O(1) dictionary/set membership, and why an index is preparation paid once.
- Uses only `pandas` and Python's built-in `sqlite3` — no installs, no network, no API keys.
- Run top to bottom — later cells depend on earlier ones.

**Suggested sessions:** (1) Units 1-2 — the pain, then the schema · (2) Units 3-4 — Python and SQLite, then when to use which

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

# Every database and fixture the notebook writes goes under ../data/ (gitignored),
# never into the repo tree. pandas writes files but not folders, so make it once.
Path("../data").mkdir(parents=True, exist_ok=True)

print("pandas        :", pd.__version__)
print("SQLite library:", sqlite3.sqlite_version)   # the engine, bundled with Python
print("working dir   :", Path.cwd().name)

## Unit 1 — The pain of files  ·  ~40 min

A file is bytes plus an agreed encoding. That is its strength — universal, portable,
readable — and its whole weakness: **bytes cannot make a promise.** A CSV cannot insist that a
column stays text, that a customer_id never repeats, that a country is never blank, or that
two files agree with each other. Lesson 2 met all of these one at a time; here we line them up
on purpose, because every one is a *property* the file failed to keep — and Unit 2 is where we
finally get to enforce it.

Everything below writes small, clearly-labelled teaching fixtures to `../data/` — the same "build
the problem so it shows up every time" habit from Lesson 2.

### 1.1 The leading-zero trap — persistence without a promise

You have seen this one. We write an id with leading zeros, save it, and read it back.

In [ ]:
people = pd.DataFrame({"customer_id": ["00123", "00204", "00987"], "spend": [40, 55, 12]})

trap_path = Path("../data/lesson7_leading_zero.csv")
people.to_csv(trap_path, index=False)
print("what we wrote to disk:")
print(trap_path.read_text())

back = pd.read_csv(trap_path)          # a naive read, exactly like Lesson 2
print("what pandas read back:")
print(back)
print("dtype of customer_id:", back["customer_id"].dtype, "| first value:", repr(back["customer_id"].iloc[0]))

🧭 We wrote `00123`; the file handed back the number `123`. This is Lesson 2's ZIP-code trap
exactly — and notice *where* it happened. The file did not lie; it simply **has no place to
record that `customer_id` is a label, not a quantity.** Lesson 2's fix was to re-declare the
type on the way in (`dtype={"customer_id": "string"}`) — a *repair*, applied by whoever
remembered to. Nothing on disk forces it. Hold that word *forces*.

### 1.2 One customer, two files, two countries

Real organisations keep the same customer in several files — a sales export here, a CRM dump
there. Watch what a file cannot do when they disagree.

In [ ]:
# Two separate files, same customer_id, contradicting each other. (Authored teaching fixture.)
pd.DataFrame({"customer_id": ["00123"], "country": ["United Kingdom"]}).to_csv(
    "../data/lesson7_country_sales.csv", index=False)
pd.DataFrame({"customer_id": ["00123"], "country": ["France"]}).to_csv(
    "../data/lesson7_country_crm.csv", index=False)

sales = pd.read_csv("../data/lesson7_country_sales.csv", dtype={"customer_id": "string"})
crm   = pd.read_csv("../data/lesson7_country_crm.csv", dtype={"customer_id": "string"})

print("sales file says customer", sales.loc[0, "customer_id"], "->", sales.loc[0, "country"])
print("crm   file says customer", crm.loc[0, "customer_id"],   "->", crm.loc[0, "country"])
print("same customer_id, different country, zero complaints from either file.")

🧭 Same `customer_id`, two different countries, and **not a whisper of complaint** from either
file. Neither one can say "there is exactly one true country per customer, and it lives here."
A file is an island; it knows nothing about any other file. What we want is a single place that
*owns* the customer record and refuses to hold two — that place is a table with a **primary
key**, and it is the first thing we build in Unit 2.

### 1.3 "This column must never be empty"

A shop can survive a missing phone number. It cannot survive a transaction with no customer, or
a customer with no country. Try to forbid a blank country using only a file.

In [ ]:
# Row 00204 is missing its country. A file stores the gap without objection.
missing = pd.DataFrame({
    "customer_id": ["00123", "00204", "00987"],
    "country":     ["United Kingdom", None, "EIRE"],
})
missing.to_csv("../data/lesson7_missing_country.csv", index=False)
print("what landed on disk:")
print(Path("../data/lesson7_missing_country.csv").read_text())

loaded = pd.read_csv("../data/lesson7_missing_country.csv", dtype={"customer_id": "string"})
print("rows with no country:", loaded["country"].isna().sum(), "(the file accepted it anyway)")

🧭 The file swallowed the blank without blinking. Lesson 2 could only *find* it afterwards with
`isna().sum()` and then decide what to do — a repair, once again, after the bad data was already
saved. There is no CSV syntax for "this column is required." `NOT NULL` is that syntax, and a
database applies it at the moment of writing, not the morning after.

### What persistence really means

"I saved a file" gets you **durability**: the data outlives the program that made it. That is
necessary and nowhere near enough. A serious data store adds two things a bare file never will:

- **Integrity** — a referee that checks every write against the rules and *rejects* violations,
  so the bad row never lands. (Lesson 2 *cleaned* data; a database keeps it from getting dirty.)
- **Coordination** — many readers and writers at once, without two of them clobbering each
  other's edits.

Persistence *plus* integrity *plus* coordination is what "database" means. The rest of this
lesson is those three words made concrete — starting with the referee.

## Unit 2 — Tables, schemas, keys  ·  ~50 min

`sqlite3` is part of Python's standard library — there is nothing to install. It runs an entire
SQL database *inside a single file*, which is why it powers phones, browsers, and aircraft, and
why it is perfect for learning: no server, no configuration, just a file in `../data/`.

The one idea of this unit: **a schema is the property checklist from Lesson 1, written in a
language the software enforces.** When you `CREATE TABLE`, you are not only naming columns — you
are declaring the rules and handing the database the authority to reject anything that breaks
them.

In [ ]:
DB = Path("../data/lesson7_demo.sqlite")
if DB.exists():
    DB.unlink()                 # start clean so re-running the notebook is repeatable

conn = sqlite3.connect(DB)      # opens (or creates) the database file

# SURPRISE: SQLite ships with foreign-key checking turned OFF, per connection.
# You must switch it on every time, or the relationship rules in 2.3 are silently ignored.
conn.execute("PRAGMA foreign_keys = ON")

print("database file :", DB)
print("foreign keys  :", conn.execute("PRAGMA foreign_keys").fetchone()[0], "(1 = enforced)")

### 2.1 CREATE TABLE — writing the checklist down

Read this statement as a list of Lesson 1 properties:

- `customer_id TEXT PRIMARY KEY` — a label (so text; leading zeros are safe) that must be *unique*.
- `country TEXT NOT NULL` — required; the blank from 1.3 is now illegal.
- `signup_date TEXT` — optional; no promise made, so none enforced.

In [ ]:
conn.execute("""
    CREATE TABLE customers (
        customer_id TEXT PRIMARY KEY,   -- unique label, never a number
        country     TEXT NOT NULL,      -- required: no blank countries
        signup_date TEXT                -- optional
    )
""")

conn.execute(
    "INSERT INTO customers (customer_id, country, signup_date) VALUES (?, ?, ?)",
    ("00123", "United Kingdom", "2010-12-01"),
)
conn.commit()

print(pd.read_sql("SELECT * FROM customers", conn))

🧭 That `CREATE TABLE` *is* the property checklist. From now on the database — not your memory,
not a comment, not a downstream `assert` — guarantees those rules for every row that ever enters
`customers`. Watch it do exactly that.

### 2.2 The database rejects a bad row

This is the heart of the lesson. Everything a file could not refuse in Unit 1, the schema
refuses now — at the moment of writing, with a loud, specific error. We wrap each attempt in
`try/except sqlite3.IntegrityError` so we can *catch the refusal and read it out loud*.

In [ ]:
# (a) A DUPLICATE primary key -- the "two countries for one customer" problem from 1.2.
try:
    conn.execute("INSERT INTO customers VALUES (?, ?, ?)", ("00123", "France", "2011-01-01"))
except sqlite3.IntegrityError as err:
    print("duplicate customer rejected ->", err)

# (b) A BLANK required column -- the missing country from 1.3.
try:
    conn.execute("INSERT INTO customers VALUES (?, ?, ?)", ("00204", None, "2011-02-01"))
except sqlite3.IntegrityError as err:
    print("empty country rejected      ->", err)

print()
print("customers is still exactly one clean row -- the damage never landed:")
print(pd.read_sql("SELECT * FROM customers", conn))

🧭 Two bad rows, two refusals, and the table stayed clean. Notice what `PRIMARY KEY` is really
doing: to reject a duplicate *instantly on every insert*, the database keeps an index of the
keys it has already seen and asks "is `00123` in here?" That is **Lesson 4's O(1) membership
test — now durable.** The set-of-seen-keys you built by hand to speed up a join is the same
structure a primary key maintains for you permanently, as a rule rather than an optimisation.

### 2.3 FOREIGN KEY — an enforced relationship

Lesson 2 linked two tables by `customer_id` and merged on it. But a merge is *hopeful*: it
happily kept transaction `T006`, whose customer `00999` did not exist, as a row full of `NaN`. A
**foreign key** turns that hope into a guarantee — a transaction may only name a customer that
actually exists.

In [ ]:
conn.execute("""
    CREATE TABLE transactions (
        transaction_id TEXT PRIMARY KEY,
        customer_id    TEXT NOT NULL,
        amount         REAL NOT NULL,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    )
""")

# A transaction for a customer who exists -> accepted.
conn.execute("INSERT INTO transactions VALUES (?, ?, ?)", ("T001", "00123", 9.95))
conn.commit()

# A transaction for customer 00999, who is NOT in customers -> rejected.
try:
    conn.execute("INSERT INTO transactions VALUES (?, ?, ?)", ("T002", "00999", 5.00))
except sqlite3.IntegrityError as err:
    print("orphan transaction rejected ->", err)

print("transactions that made it in:")
print(pd.read_sql("SELECT * FROM transactions", conn))

🧭 The exact row a left join would have kept as `NaN`, the database threw out at the door. That
is the difference between *analysing* a broken relationship and *forbidding* it. (And remember
the gotcha: had we skipped `PRAGMA foreign_keys = ON`, SQLite would have accepted the orphan
silently — the rule is only as good as the switch that turns it on.)

### 2.4 One honest caveat — types are affinities, not always walls

SQLite enforces `NOT NULL`, `PRIMARY KEY`, and `FOREIGN KEY` strictly. It is deliberately
*lenient* about column types: by default it will happily store text in an `INTEGER` column. If
you want types enforced too, declare the table `STRICT`.

In [ ]:
conn.execute("CREATE TABLE loose (n INTEGER)")
conn.execute("INSERT INTO loose VALUES ('banana')")      # accepted, despite INTEGER
print("plain table stored:", conn.execute("SELECT n FROM loose").fetchone())

conn.execute("CREATE TABLE strict_demo (n INTEGER) STRICT")
try:
    conn.execute("INSERT INTO strict_demo VALUES ('banana')")
except sqlite3.IntegrityError as err:
    print("STRICT table rejected the wrong type ->", err)

conn.close()   # done with the teaching database

🧭 So a schema enforces *most* of Lesson 1's checklist out of the box — required-ness,
uniqueness, relationships — and the rest (exact types) when you ask for `STRICT`.

> **Unit 2 takeaway:** `CREATE TABLE` is the property checklist made executable. `NOT NULL`,
> `PRIMARY KEY`, and `FOREIGN KEY` are three promises the database keeps on *every write*, by
> rejecting anything that would break them — the referee that files never had.

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. Name three properties a plain CSV file cannot enforce.
2. "I saved a file" gives you durability. What two things does a database add on top?
3. What does `PRIMARY KEY` guarantee, and which Lesson 4 idea makes that check cheap?
4. A `FOREIGN KEY` would have refused Lesson 2's orphan transaction. What SQLite gotcha could
   let the orphan slip in anyway?
5. SQLite stored `'banana'` in an `INTEGER` column without complaint. What does that tell you
   about SQLite's types, and how do you make types strict?

<details><summary><b>Answers</b></summary>

1. Any three: a column must stay one type; an id must be unique; a column must not be blank; two
   files must agree. (Files enforce none of them.)
2. Integrity (a referee that rejects rule-breaking rows on write) and coordination (many
   readers/writers at once).
3. Uniqueness of the key — no two rows share it. The database keeps an index of seen keys and
   tests membership in O(1) average, Lesson 4's dictionary/set trick made durable.
4. Foreign-key checking is OFF by default, per connection; you must run
   `PRAGMA foreign_keys = ON` every time, or the rule is silently ignored.
5. SQLite is lenient about column types (type *affinity*) but strict about NOT NULL / PRIMARY
   KEY / FOREIGN KEY. Declare the table `STRICT` to enforce types too.
</details>

## Unit 3 — Python to SQLite and back  ·  ~45 min

You will rarely type SQL by hand into a database. You will drive it from Python: open a
connection, hand data in, pull results back into a DataFrame. Now we do it with the **real**
two-table retail extract from Lesson 2 — the same 10 customers and 60 transactions.

In [ ]:
customers = pd.read_csv("../course_data/lesson2_customers_base.csv")
transactions = pd.read_csv("../course_data/lesson2_transactions_base.csv")
print("from disk:", len(customers), "customers,", len(transactions), "transactions")

RETAIL = Path("../data/lesson7_retail.sqlite")
if RETAIL.exists():
    RETAIL.unlink()
retail = sqlite3.connect(RETAIL)

# df.to_sql writes a whole DataFrame into a table in one call.
customers.to_sql("customers", retail, if_exists="replace", index=False)
transactions.to_sql("transactions", retail, if_exists="replace", index=False)

print("tables now in the database:",
      [row[0] for row in retail.execute("SELECT name FROM sqlite_master WHERE type='table'")])

🧭 `to_sql` is the fast way in — and a quiet warning about our through-line. It *invented* a
schema from the DataFrame's dtypes, and that schema has **no primary key, no NOT NULL, no
foreign key.** Convenience threw the checklist away. Look:

In [ ]:
# The schema to_sql generated. Field order: (cid, name, type, notnull, default, pk).
print("(cid, name, type, notnull, default, pk)")
for column in retail.execute("PRAGMA table_info(customers)"):
    print(column)

🧭 Every `notnull` and `pk` flag is `0` — the DataFrame -> SQLite trip lost exactly the
properties Unit 2 spent its time enforcing. This is the same lesson as CSV amnesia in Lesson 2,
and it is precisely why the exercise makes *you* write the `CREATE TABLE` first and load into
it, rather than letting `to_sql` guess.

### 3.1 Reading results back with pd.read_sql

The database is a fine calculator. Ask it for revenue per customer and it hands the answer
straight back as a DataFrame — the round trip closes.

In [ ]:
revenue_by_customer = pd.read_sql(
    """
    SELECT customer_id,
           ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM transactions
    GROUP BY customer_id
    ORDER BY revenue DESC
    """,
    retail,
)
print(revenue_by_customer)

🧭 That `GROUP BY` is Lesson 2's `groupby` — *split, apply, combine* — run by the database
engine instead of by pandas. Same algorithm (Lesson 4: one scan carrying a running total per
key), different engine. You now have two ways to compute the identical answer, which is the
whole idea behind the Lesson 9 assignment.

### 3.2 Cursors — the row-by-row API

`read_sql` is perfect when you want a table back. When you want to *do something* with each
result, use a **cursor**: `execute` runs the SQL, `fetchall` collects the rows.

In [ ]:
cur = retail.cursor()
cur.execute(
    "SELECT transaction_id, product FROM transactions WHERE customer_id = ?",
    ("C15311",),                 # the value goes in the tuple, NOT in the string
)
rows = cur.fetchall()
print(f"{len(rows)} transactions for C15311; first three:")
for transaction_id, product in rows[:3]:
    print("  ", transaction_id, "-", product)

### 3.3 The `?` placeholder — the only safe way to pass a value

Notice we never pasted `C15311` into the SQL string; we passed it as a parameter in a tuple.
That `?` is not a style preference — it is the rule. Here is what happens when you break it,
using a real product name that contains an apostrophe.

In [ ]:
messy_value = "YOU'RE CONFUSING ME METAL SIGN"   # a genuine product name in our data

# WRONG: paste the value straight into the query string.
pasted = f"SELECT * FROM transactions WHERE product = '{messy_value}'"
print("the string we built:")
print(" ", pasted)
try:
    retail.execute(pasted).fetchall()
except sqlite3.OperationalError as err:
    print("  -> the database choked:", err)

# RIGHT: pass the value as a parameter and let the driver handle the quoting.
safe = retail.execute(
    "SELECT transaction_id FROM transactions WHERE product = ?",
    (messy_value,),
).fetchall()
print("parameterized query worked:", safe)

🧭 The apostrophe in `YOU'RE` closed the string early, and the rest became broken SQL. That is
the *harmless* version. The dangerous version is when the value comes from someone who *wants* to
break it: a crafted input like `'; DROP TABLE customers; --` turns a pasted query into two
commands, the second of which deletes your table. This is **SQL injection**, and the fix is
always the same and always easy — never build queries with `+` or f-strings; pass every value as
a `?` parameter and let the driver quote it safely.

> **Unit 3 takeaway:** `to_sql` / `read_sql` move whole tables between pandas and SQLite; cursors
> walk rows; and every value you inject goes through a `?` placeholder — for correctness today
> and safety always.

## Unit 4 — Database vs DataFrame  ·  ~30 min

You now hold two tools that can answer the same question. When do you reach for which? They are
not rivals; they are good at different things.

| | **SQLite database** | **pandas DataFrame** |
| --- | --- | --- |
| Lives in | a file on disk | memory (RAM) |
| Survives a restart? | yes — durable | no — gone with the kernel |
| Enforces the schema? | yes — NOT NULL, keys, relationships | no — you check by hand |
| Many writers at once? | yes — coordinated | no — one process |
| Bigger than RAM? | fine — reads only what it needs | must fit in memory |
| Fast to reshape and explore? | SQL, less nimble | its whole reason to exist |
| Plot, model, `scikit-learn`? | not directly | yes — the analytics home |

In [ ]:
# The SAME revenue answer, computed by each engine, must agree.
by_sql = pd.read_sql(
    "SELECT customer_id, ROUND(SUM(quantity * unit_price), 2) AS revenue "
    "FROM transactions GROUP BY customer_id",
    retail,
).sort_values("customer_id").reset_index(drop=True)

by_pandas = (
    transactions.assign(revenue=transactions["quantity"] * transactions["unit_price"])
    .groupby("customer_id")["revenue"].sum().round(2)
    .reset_index().sort_values("customer_id").reset_index(drop=True)
)

print("SQL and pandas agree:", by_sql.equals(by_pandas))
print(by_sql.head(3))

retail.close()   # close the database connection when the work is done

🧭 Identical numbers, because it is the identical algorithm — the database and pandas just run it
in different places. That is exactly the Lesson 9 assignment in miniature: *one analysis, two
engines.* The choice between them is never about the answer; it is about durability, enforcement,
size, and who else needs the data.

### When a CSV (or DataFrame) is genuinely fine

Not everything needs a database. Reach for a plain file when:

- the data is **small and fits in memory**, and you are exploring or prototyping;
- there is **one author** and no concurrent writers to coordinate;
- it is a **one-off** analysis, or a hand-off to a human (a CSV or Excel they can open);
- the properties barely matter because the data is throwaway.

Reach for a database when the data must **outlive the session, be trusted by many people, enforce
its own rules, or grow past memory** — a customer list, an orders ledger, anything where a single
bad row is a real problem. The honest summary: *a DataFrame is a fast workbench; a database is a
durable system of record.*

## Wrap-up

### Five misconceptions this lesson should have broken

| Misconception | Reality |
| --- | --- |
| "Saving a file protects my data" | a file gives durability only; it enforces no property and refuses no bad row |
| "A schema is just column names" | it is Lesson 1's checklist made executable — types, required-ness, uniqueness, relationships |
| "The database trusts what I insert" | it rejects duplicates, blanks, and orphans on write — the emotional core of Unit 2 |
| "Foreign keys are on by default" | SQLite ships with them OFF; `PRAGMA foreign_keys = ON`, per connection |
| "`to_sql` gives me a real schema" | it invents one with no keys and no NOT NULL — you must declare the rules yourself |

### The lesson, one sentence

> *A schema is the property checklist from Lesson 1, handed to software that enforces it on every
> write — so integrity stops being a discipline you hope to keep and becomes a rule the database
> keeps for you.*

### Where each Lesson 1 property finally got enforced

- *type attached to the value* -> column types (and `STRICT` when you want a hard wall)
- *uniqueness (sets, dict keys)* -> `PRIMARY KEY`, backed by Lesson 4's O(1) membership index
- *values not blank / within a legal set* -> `NOT NULL` (and `CHECK`, its cousin)
- *keys -> records (dictionaries)* -> `FOREIGN KEY`, the merge key turned into a guarantee
- *"storage is not neutral"* -> a database chooses to preserve properties a CSV throws away

### What you can now do

- Explain, with live demos, why files cannot enforce integrity and a database can.
- Write `CREATE TABLE` with `NOT NULL`, `PRIMARY KEY`, and `FOREIGN KEY`, and predict which rows
  it will reject.
- Move data between pandas and SQLite with `to_sql` and `read_sql`, and query it safely with `?`
  parameters.
- Choose between a database and a DataFrame for a given job, and say why.

### Practice

Open **`../lesson7_exercise/`**: design the two-table retail schema yourself, load the real
customers and transactions into it, and make the database reject one bad row on purpose. A pytest
checker inspects your schema and your rejection — `README.md` in that folder has the exact
commands.

**Next:** Lesson 8 takes SQL from "store and fetch" to "ask real questions" — `WHERE`,
`GROUP BY`, `NULL`, and evaluation order. Lesson 9 adds joins and window functions, then
Assignment A3 runs *one analysis on two engines* so this database becomes the source of truth.

### Extensions

- [SQLite documentation](https://www.sqlite.org/docs.html) — the official reference for the engine
- [SQLite Tutorial](https://www.sqlitetutorial.net/) — a gentle, example-driven walkthrough of SQL on SQLite
- [Python `sqlite3` module](https://docs.python.org/3/library/sqlite3.html) — the driver you used here, with its own tutorial section